## Estado de este notebook

Las cifras que este notebook produjo originalmente se obtuvieron con una **particion
aleatoria estratificada**, que repartia descripciones identicas entre entrenamiento y
prueba. La medicion resultaba optimista en unos tres puntos en todas las configuraciones.

Desde el 14 de agosto la evaluacion usa una **particion agrupada por descripcion**
(fuga 0.00 %). La celda de particion de abajo ya toma esa version, importandola de
`lab/scripts/eval_grouped_split.py`, que es la unica fuente de la particion.

**Las salidas de las celdas afectadas se limpiaron**: mostrar las anteriores seria
contradecir las cifras del informe. Hay que reejecutar el notebook para repoblarlas.

Los resultados vigentes de la competencia viven en `lab/results/grouped_split.json`
y se reproducen con:

```
python3 ../scripts/eval_grouped_split.py                 # las 6 configuraciones clasicas
python3 ../scripts/train_transformer.py --split grouped  # el transformer, por separado
```


# Competencia de Modelos v2: Nuevos Candidatos

**Objetivo**: Evaluar 3 modelos adicionales contra el baseline ganador (LinearSVC + CharTFIDF, accuracy 0.8491).

**Nuevos candidatos**:
1. **XGBoost** + CharTFIDF
2. **fastText** (subword embeddings nativos)
3. **Fine-tuned Transformer** (multilingual-MiniLM)

Se reutiliza la misma carga de datos y preprocesamiento del notebook 02.

## 1. Setup

In [ ]:
!pip install xgboost fasttext-wheel transformers torch datasets scikit-learn pandas matplotlib seaborn openpyxl


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
import re
import glob
import time
import tempfile
import warnings
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, top_k_accuracy_score
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 80)

if os.path.exists('/home/jovyan/work/data/xlsx'):
    DATA_DIR = '/home/jovyan/work/data/xlsx'
else:
    DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'xlsx')

print(f'DATA_DIR: {DATA_DIR}')
print('Imports OK')

DATA_DIR: /Users/menene/code/www/mbia_gamma/lab/data/xlsx
Imports OK


## 2. Carga de Datos (identica a notebook 02)

In [2]:
def _to_str(val):
    if val is None or pd.isna(val):
        return None
    if isinstance(val, float) and val == int(val):
        return str(int(val))
    return str(val).strip()

def _extract_code(val):
    if val is None or pd.isna(val):
        return None
    s = str(val).strip()
    if ' - ' in s:
        return s.split(' - ')[0].strip()
    return s

def _prefix_rename(df, col_map):
    rename = {}
    used_dsts = set()
    for col in df.columns:
        for src, dst in col_map.items():
            if dst in used_dsts:
                continue
            if col == src or col.startswith(src):
                rename[col] = dst
                used_dsts.add(dst)
                break
    return df.rename(columns=rename)

def preprocess_text(text: str) -> str:
    text = text.upper()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[;:,/\\|]+', ' ', text)
    text = re.sub(r'[^A-Z0-9.\-\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [3]:
# Cargar clases
CLASS_COLS = {
    'Código': 'class_code', 'Denominación': 'class_name',
    'Grupo de Artículos': 'article_group', 'Sector': 'sector',
    'Tipo de Material': 'material_type', 'UNSPSC': 'unspsc',
}
all_files = glob.glob(os.path.join(DATA_DIR, '*.xlsx'))
classes_files = [f for f in all_files if 'clase' in os.path.basename(f).lower()]
material_files = [f for f in all_files if 'clase' not in os.path.basename(f).lower()
                  and 'unspsc' not in os.path.basename(f).lower()
                  and 'naciones' not in os.path.basename(f).lower()]

dfs_classes = []
for f in classes_files:
    df_c = pd.read_excel(f, dtype=str, engine='openpyxl')
    df_c = _prefix_rename(df_c, CLASS_COLS)
    dfs_classes.append(df_c)
df_classes = pd.concat(dfs_classes, ignore_index=True)
df_classes['class_code'] = df_classes['class_code'].apply(_to_str)
if 'material_type' in df_classes.columns:
    df_classes['material_type'] = df_classes['material_type'].apply(_extract_code)
df_classes = df_classes.dropna(subset=['class_code', 'class_name']).drop_duplicates(subset=['class_code'])

# Cargar materiales
MATERIAL_COLS = {
    'Material': 'material_code', 'Unidad medida base': 'uom',
    'Denom.estándar': 'class_code', 'Texto breve de material': 'short_text',
}
dfs_materials = []
for f in material_files:
    df_m = pd.read_excel(f, dtype=str, engine='openpyxl')
    df_m = _prefix_rename(df_m, MATERIAL_COLS)
    keep = [c for c in ['material_code', 'class_code', 'short_text'] if c in df_m.columns]
    df_m = df_m[keep].copy()
    dfs_materials.append(df_m)
df_materials = pd.concat(dfs_materials, ignore_index=True)
df_materials['material_code'] = df_materials['material_code'].apply(_to_str)
df_materials['class_code'] = df_materials['class_code'].apply(_to_str)
df_materials = df_materials.dropna(subset=['material_code', 'short_text'])

# Cruzar
df = df_materials.merge(
    df_classes[['class_code', 'class_name', 'material_type']],
    on='class_code', how='inner'
)
df = df.loc[:, ~df.columns.duplicated()]
df['clean_text'] = df['short_text'].apply(preprocess_text)

# Filtrar clases con <3 ejemplos
MIN_SAMPLES = 3
valid_classes = df['class_code'].value_counts()
valid_classes = valid_classes[valid_classes >= MIN_SAMPLES].index
df_model = df[df['class_code'].isin(valid_classes)].copy()

print(f'Dataset: {len(df_model):,} materiales, {df_model["class_code"].nunique()} clases')

Dataset: 39,571 materiales, 1234 clases


In [ ]:
# Particion agrupada por descripcion. Se toma de lab/scripts/eval_grouped_split.py
# para que este notebook, el script de la competencia y el entrenamiento del
# transformer usen exactamente las mismas fronteras.
#
# Antes aqui habia un train_test_split estratificado. Repartia descripciones
# identicas entre entrenamiento y prueba —25.72 % de las filas de test se veian
# tambien en train— e inflaba todas las metricas alrededor de tres puntos.
import sys
sys.path.append(os.path.join('..', 'scripts'))
import eval_grouped_split as egs

df_model = egs.load_dataset()
X, y, le, splits = egs.make_splits(df_model)
tr, te = splits['grouped']
X_train, X_test = X[tr], X[te]
y_train, y_test = y[tr], y[te]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | Clases: {len(le.classes_)}')
print('Protocolo: grouped (fuga 0.00 %)')


## 3. Evaluacion comun

In [5]:
results = {}

# Baseline del notebook 02 (para comparar sin reentrenar)
results['LinearSVC + CharTFIDF (baseline)'] = {
    'accuracy': 0.8491,
    'f1_macro': 0.7523,
    'f1_weighted': 0.8380,
    'precision_weighted': 0.8419,
    'recall_weighted': 0.8491,
    'top3_accuracy': 0.9404,
    'train_time': 62.9,
}

def eval_model(name, y_pred, y_proba=None, train_time=0):
    """Evalua predicciones y guarda resultados."""
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)

    top3 = None
    if y_proba is not None:
        k = min(3, y_proba.shape[1])
        top3 = top_k_accuracy_score(y_test, y_proba, k=k, labels=range(y_proba.shape[1]))

    results[name] = {
        'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted,
        'precision_weighted': prec, 'recall_weighted': rec,
        'top3_accuracy': top3, 'train_time': train_time,
        'y_pred': y_pred,
    }

    print(f'  Accuracy:       {acc:.4f}')
    print(f'  F1 (macro):     {f1_macro:.4f}')
    print(f'  F1 (weighted):  {f1_weighted:.4f}')
    print(f'  Precision (w):  {prec:.4f}')
    print(f'  Recall (w):     {rec:.4f}')
    if top3 is not None:
        print(f'  Top-3 Acc:      {top3:.4f}')
    print(f'  Tiempo:         {train_time:.1f}s')

## 4. Modelo 1: XGBoost + CharTFIDF

Gradient boosting sobre los mismos character n-gram features. XGBoost suele superar a SVMs
en problemas tabulares y escala bien a muchas clases con su implementacion multi-class nativa.

In [ ]:
from xgboost import XGBClassifier

print('Vectorizando (CharTFIDF)...')
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5),
    max_features=50000, sublinear_tf=True, strip_accents='unicode'
)
X_train_tfidf = tfidf_char.fit_transform(X_train)
X_test_tfidf = tfidf_char.transform(X_test)

print(f'TF-IDF shape: {X_train_tfidf.shape}')
print(f'Entrenando XGBoost ({len(le.classes_)} clases)...')

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(le.classes_),
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    verbosity=1,
    eval_metric='mlogloss',
)
xgb.fit(X_train_tfidf, y_train)
xgb_time = time.time() - t0

y_pred_xgb = xgb.predict(X_test_tfidf)
y_proba_xgb = xgb.predict_proba(X_test_tfidf)

print(f'\nXGBoost + CharTFIDF:')
eval_model('XGBoost + CharTFIDF', y_pred_xgb, y_proba_xgb, xgb_time)

## 5. Modelo 2: fastText

fastText aprende embeddings de subpalabras de forma nativa (no necesita TF-IDF externo).
Extremadamente rapido de entrenar y fuerte en tareas de clasificacion de texto corto.

In [ ]:
import fasttext

# fastText necesita archivos en formato: __label__<class> <text>
train_file = os.path.join(tempfile.gettempdir(), 'ft_train.txt')
test_file = os.path.join(tempfile.gettempdir(), 'ft_test.txt')

# Mapeo label int -> class_code para fastText labels
label_to_code = {i: code for i, code in enumerate(le.classes_)}
code_to_label = {code: i for i, code in enumerate(le.classes_)}

with open(train_file, 'w') as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{label_to_code[label]} {text}\n')

with open(test_file, 'w') as f:
    for text, label in zip(X_test, y_test):
        f.write(f'__label__{label_to_code[label]} {text}\n')

print(f'Archivos escritos: train={len(X_train):,}, test={len(X_test):,}')
print('Entrenando fastText...')

t0 = time.time()
ft_model = fasttext.train_supervised(
    input=train_file,
    epoch=50,
    lr=0.5,
    wordNgrams=2,
    minn=2,
    maxn=5,
    dim=100,
    loss='softmax',
    bucket=2000000,
    thread=os.cpu_count(),
    verbose=2,
)
ft_time = time.time() - t0

print(f'Entrenamiento: {ft_time:.1f}s')

In [10]:
import fasttext                                                                                                          
import numpy as np
                                                                                                                           
_original_predict = fasttext.FastText._FastText.predict                                                                

def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error="strict"):
    labels, probs = _original_predict(self, text, k, threshold, on_unicode_error)
    return labels, np.asarray(probs)

fasttext.FastText._FastText.predict = _patched_predict

In [12]:
import fasttext.FastText as _ft_module
import numpy as np                                                                                                       
                                                                                                                        
_src = _ft_module._FastText.predict
import types, inspect

# Patch numpy directly in fasttext's module namespace
_ft_module.np.array = lambda obj, **kw: np.asarray(obj)

In [ ]:
# Evaluar fastText
n_classes = len(le.classes_)

y_pred_ft = []
y_proba_ft = np.zeros((len(X_test), n_classes))

for i, text in enumerate(X_test):
    preds = ft_model.predict(text, k=n_classes)
    labels, probs = preds

    # Top-1 prediction
    top_code = labels[0].replace('__label__', '')
    y_pred_ft.append(code_to_label[top_code])

    # Fill probability matrix
    for lbl, prob in zip(labels, probs):
        code = lbl.replace('__label__', '')
        if code in code_to_label:
            y_proba_ft[i, code_to_label[code]] = prob

y_pred_ft = np.array(y_pred_ft)

print('fastText:')
eval_model('fastText', y_pred_ft, y_proba_ft, ft_time)

## 6. Modelo 3: Fine-tuned Transformer (multilingual-MiniLM)

Un transformer pequeno (22M params) pre-entrenado en 100+ idiomas. Lo fine-tuneamos
directamente sobre los short_texts. Es el modelo mas pesado de entrenar pero captura
semantica que TF-IDF no puede.

In [ ]:
!pip install sentencepiece tiktoken

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "sentencepiece", "protobuf"])

In [21]:
import sentencepiece

In [ ]:
import importlib                                                                                                                                                  
import transformers.utils.import_utils as _iu
                                                                                                                                                                
# Force re-check of sentencepiece availability                                                                                                                  
importlib.reload(_iu)

# Also reload the modules that use it
import transformers.convert_slow_tokenizer as _cst
importlib.reload(_cst)
import transformers.tokenization_utils_tokenizers as _tut
importlib.reload(_tut)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

DEVICE = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE}')

MODEL_NAME = 'microsoft/Multilingual-MiniLM-L12-H384'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer cargado: {MODEL_NAME}')

In [ ]:
class MaterialDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }

BATCH_SIZE = 64
MAX_LEN = 64
EPOCHS = 5
LR = 2e-5

train_ds = MaterialDataset(X_train.tolist(), y_train.tolist(), tokenizer, MAX_LEN)
test_ds = MaterialDataset(X_test.tolist(), y_test.tolist(), tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le.classes_)
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

print(f'Parametros: {sum(p.numel() for p in model.parameters()):,}')
print(f'Entrenando {EPOCHS} epochs...')

t0 = time.time()
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item() * labels.size(0)
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = epoch_loss / total
    train_acc = correct / total
    print(f'  Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f}, train_acc={train_acc:.4f}')

transformer_time = time.time() - t0
print(f'\nEntrenamiento completo: {transformer_time:.1f}s')

In [ ]:
# Evaluar transformer
model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = probs.argmax(dim=-1)

        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

y_pred_tf = np.concatenate(all_preds)
y_proba_tf = np.concatenate(all_probs)

print('Transformer (multilingual-MiniLM):')
eval_model('Transformer (MiniLM)', y_pred_tf, y_proba_tf, transformer_time)

## 7. Comparacion

In [ ]:
comparison = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'F1 Macro': r['f1_macro'],
        'F1 Weighted': r['f1_weighted'],
        'Precision': r['precision_weighted'],
        'Recall': r['recall_weighted'],
        'Top-3 Acc': r['top3_accuracy'] if r['top3_accuracy'] else float('nan'),
        'Tiempo (s)': r['train_time'],
    }
    for name, r in results.items()
}).T

comparison_styled = comparison.style.highlight_max(
    axis=0, subset=['Accuracy', 'F1 Macro', 'F1 Weighted', 'Precision', 'Recall', 'Top-3 Acc'],
    color='lightgreen'
).format('{:.4f}')
comparison_styled

In [30]:
import numpy as np                                                                                                       
np.array = np._core.multiarray.array

In [ ]:
# Grafico comparativo
metrics_to_plot = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'Precision', 'Recall']
fig, ax = plt.subplots(figsize=(14, 6))

model_names = [n for n in results.keys()]
x = np.arange(len(metrics_to_plot))
width = 0.18
colors = sns.color_palette('Set2', len(model_names))

for i, name in enumerate(model_names):
    vals = [comparison.loc[name, m] for m in metrics_to_plot]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])

ax.set_xticks(x + width * (len(model_names) - 1) / 2)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score')
ax.set_title('Comparacion de Modelos v2')
ax.legend(loc='lower right', fontsize=8)
ax.axhline(y=0.85, color='gray', linestyle='--', alpha=0.5, label='Baseline')
plt.tight_layout()
plt.show()

## 8. Matriz de Confusion del Mejor Modelo Nuevo

In [ ]:
# Identificar el mejor modelo nuevo (excluyendo baseline)
new_models = {k: v for k, v in results.items() if 'baseline' not in k}
best_name = max(new_models, key=lambda k: new_models[k]['f1_weighted'])
best = new_models[best_name]

print(f'Mejor modelo nuevo: {best_name}')
print(f'  Accuracy:     {best["accuracy"]:.4f}')
print(f'  F1 Weighted:  {best["f1_weighted"]:.4f}')
print(f'  F1 Macro:     {best["f1_macro"]:.4f}')
if best['top3_accuracy']:
    print(f'  Top-3 Acc:    {best["top3_accuracy"]:.4f}')

# Matriz de confusion top 20 clases
TOP_N = 20
top_classes = pd.Series(y_test).value_counts().head(TOP_N).index.tolist()
mask = np.isin(y_test, top_classes)

cm = confusion_matrix(y_test[mask], best['y_pred'][mask], labels=top_classes)
class_code_map = df_model.drop_duplicates('label').set_index('label')['class_name'].to_dict()
top_labels = [class_code_map.get(c, str(c))[:25] for c in top_classes]

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=top_labels, yticklabels=top_labels,
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusion - Top {TOP_N} Clases ({best_name})')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report top 20
report = classification_report(
    y_test[mask], best['y_pred'][mask],
    labels=top_classes, target_names=top_labels, zero_division=0
)
print(f'Classification Report - Top {TOP_N} ({best_name})')
print(report)

## 9. Analisis de Confianza del Mejor Modelo

In [ ]:
# Comparar umbrales de confianza para todos los modelos nuevos
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for name, r in results.items():
    if 'baseline' in name or 'y_pred' not in r:
        continue

    # Buscar la proba matrix
    if name == 'XGBoost + CharTFIDF':
        proba = y_proba_xgb
    elif name == 'fastText':
        proba = y_proba_ft
    elif name == 'Transformer (MiniLM)':
        proba = y_proba_tf
    else:
        continue

    confs = proba.max(axis=1)
    print(f'\n{name}:')
    for t in thresholds:
        m = confs >= t
        if m.sum() > 0:
            acc = accuracy_score(y_test[m], r['y_pred'][m])
            print(f'  Umbral {t:.1f}: accuracy={acc:.4f}, cobertura={m.mean():.2%} ({m.sum():,})')

## 10. Conclusion

In [ ]:
print('=' * 70)
print('RESUMEN COMPETENCIA v2')
print('=' * 70)
print(f'\nDataset: {len(df_model):,} materiales, {df_model["class_code"].nunique()} clases')
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'\nResultados:')
print(comparison.to_string())

overall_best = comparison['F1 Weighted'].idxmax()
print(f'\nMEJOR MODELO OVERALL: {overall_best}')
print(f'  Accuracy:     {comparison.loc[overall_best, "Accuracy"]:.4f}')
print(f'  F1 Weighted:  {comparison.loc[overall_best, "F1 Weighted"]:.4f}')
print(f'  F1 Macro:     {comparison.loc[overall_best, "F1 Macro"]:.4f}')
print(f'  Top-3 Acc:    {comparison.loc[overall_best, "Top-3 Acc"]:.4f}')